# 列加速(3.5.3)

収束の遅い列を、同じ極限に速く収束する列へ変換する。

例として、$\pi$に収束する交代級数の部分和を使う。

$$
\frac{\pi}{4} = 1 - \frac{1}{3} + \frac{1}{5} - \frac{1}{7} + \cdots
$$

カーネルは`Gauche (gosh)`を選ぶ。上のセルから順に実行する。

## なぜ$\frac{\pi}{4}$に収束するのか

$\arctan 1 = \frac{\pi}{4}$を級数で表したもの(ライプニッツの公式)。

1. arctanの微分は公比$-x^2$の等比級数になる。

$$
\frac{d}{dx}\arctan x = \frac{1}{1+x^2} = 1 - x^2 + x^4 - x^6 + \cdots
$$

2. 0から$x$まで項ごとに積分すると、分母の1, 3, 5, 7が出てくる。

$$
\arctan x = x - \frac{x^3}{3} + \frac{x^5}{5} - \frac{x^7}{7} + \cdots
$$

3. $\tan\frac{\pi}{4} = 1$なので、$x = 1$を代入すると$\frac{\pi}{4} = 1 - \frac{1}{3} + \frac{1}{5} - \cdots$になる。


## ストリームの基本部品

In [1]:
;; cons-stream a bは(cons a (delay b))に書き換わるというルール
(define-syntax cons-stream
  (syntax-rules ()
    ((_ a b) (cons a (delay b)))))

;; Haskellでいうhead
(define (stream-car s) (car s))

;; Haskellでいうtail
(define (stream-cdr s) (force (cdr s)))

;; n番目を取得 carでn回開く必要があるため、O(n)
(define (stream-ref s n)
  (if (= n 0)
      (stream-car s)
      (stream-ref (stream-cdr s) (- n 1))))

;; 無限ストリーム専用なので空ストリームの判定は省略している。
(define (stream-map proc s)
  (cons-stream (proc (stream-car s))
               (stream-map proc (stream-cdr s))))
;; HaskellでいうzipWith (+)
(define (add-streams s1 s2)
  (cons-stream (+ (stream-car s1) (stream-car s2))
               (add-streams (stream-cdr s1) (stream-cdr s2))))
;; Haskellでいう map (* factor)
(define (scale-stream s factor)
  (stream-map (lambda (x) (* x factor)) s))

;; Haskellでいうtake
(define (stream-head s n)
  (if (= n 0)
      '()
      (cons (stream-car s)
            (stream-head (stream-cdr s) (- n 1)))))

;; 先頭n個を1行ずつ表示する
(define (display-stream-n s n)
  (for-each (lambda (x) (display x) (newline))
            (stream-head s n)))


display-stream-n


## 元の列: `pi-stream`

`pi-summands`は$\frac{1}{1}, -\frac{1}{3}, \frac{1}{5}, \ldots$を返す。

In [2]:
(define (pi-summands n)
  (cons-stream (/ 1.0 n)
               (stream-map - (pi-summands (+ n 2))))) ;; - で符号を毎回逆にする

;; 自己参照させてpromiseを共有するので、各部分和は1回しか計算されない(O(n))。
(define (partial-sums s)
  (define ps
    (cons-stream (stream-car s)
                 (add-streams (stream-cdr s) ps)))
  ps)

(define pi-stream
  (scale-stream (partial-sums (pi-summands 1)) 4))

(display-stream-n pi-stream 8)

4.0
2.666666666666667
3.466666666666667
2.8952380952380956
3.3396825396825403
2.9760461760461765
3.2837384837384844
3.017071817071818


## オイラー変換: `euler-transform`

$S_n$を元の列の$n$番目の項とすると、加速した列の項は次のようになる。

$$
S_{n+1} - \frac{(S_{n+1} - S_n)^2}{S_{n-1} - 2S_n + S_{n+1}}
$$

連続する3項から次の値を予測する。交代級数の部分和にうまく働く。

In [3]:
(define (euler-transform s)
  (let ((s0 (stream-ref s 0))   ; S_{n-1}
        (s1 (stream-ref s 1))   ; S_n
        (s2 (stream-ref s 2)))  ; S_{n+1}
    (cons-stream (- s2 (/ (square (- s2 s1))
                          (+ s0 (* -2 s1) s2)))
                 (euler-transform (stream-cdr s)))))

(display-stream-n (euler-transform pi-stream) 8)

3.166666666666667
3.1333333333333337
3.1452380952380956
3.13968253968254
3.1427128427128435
3.1408813408813416
3.142071817071818
3.1412548236077655


8項目で小数第3位まで合うようになった。

## 加速を繰り返す: タブロー

加速した列をさらに加速できる。「列のストリーム」(タブロー)を作り、各行がその前の行を変換したものにする。

```
s00 s01 s02 s03 ...      ; 元の列
    s10 s11 s12 ...      ; 1回加速
        s20 s21 ...      ; 2回加速
            s30 ...
```

In [4]:
(define (make-tableau transform s)
  (cons-stream s
               (make-tableau transform (transform s))))

;; 各行の先頭だけを取り出した列
(define (accelerated-sequence transform s)
  (stream-map stream-car
              (make-tableau transform s)))

(display-stream-n (accelerated-sequence euler-transform pi-stream) 8)

4.0
3.166666666666667
3.142105263157895
3.141599357319005
3.1415927140337785
3.1415926539752927
3.1415926535911765
3.141592653589778


各行の先頭を取り出すだけで、8項目にして倍精度の限界近くまで$\pi$に一致する。

## 比較: 真の値との誤差

各列の$n$項目と$\pi$との差を並べる。

In [5]:
(define pi-true (* 4 (atan 1)))

(define (error-stream s)
  (stream-map (lambda (x) (abs (- x pi-true))) s))

(define (show-errors label s n)
  (display label)
  (newline)
  (display-stream-n (error-stream s) n)
  (newline))

(show-errors ";; pi-stream" pi-stream 6)
(show-errors ";; euler-transform" (euler-transform pi-stream) 6)
(show-errors ";; accelerated-sequence"
             (accelerated-sequence euler-transform pi-stream) 6)

;; pi-stream
0.8584073464102069
0.47492598692312615
0.32507401307687367
0.2463545583516975
0.19808988609274714
0.16554647754361662

;; euler-transform
0.025074013076873847
0.008259320256459368
0.0036454416483024943
0.0019101139072530415
0.0011201891230503414
7.113127084514836e-4

;; accelerated-sequence
0.8584073464102069
0.025074013076873847
5.126095681018406e-4
6.703729211743337e-6
6.044398537241591e-8
3.8549963221612416e-10



- `pi-stream`: 誤差がほぼ$\frac{1}{n}$程度でしか減らない。
- `euler-transform`: 1回の加速で、誤差が1桁以上小さくなる。
- `accelerated-sequence`: 1項ごとに誤差が数桁ずつ小さくなる。

ストリームでは、列そのものを値として`euler-transform`に渡せる。状態変数で書くと、加速を何段も重ねる処理はずっと書きにくい。